<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.1_Praktikum_Convolutional_Neural_Network.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.1: Praktikum Convolutional Neural Network (CNN)
**Tujuan Pembelajaran:**
1. Membangun dan memverifikasi operasi konvolusi 2D secara matematis murni (NumPy).
2. Mengembangkan arsitektur CNN modular menggunakan PyTorch (`torch.nn.Conv2d`, `BatchNorm2d`, `AdaptiveAvgPool2d`).
3. Mensimulasikan dataset citra patologi bercak daun kelapa sawit (*Curvularia*) dan melatih model deteksi biner hingga konvergensi.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Reproducibility seed
torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch Version: {torch.__version__}")

### Bagian 1: Verifikasi Numerik Konvolusi NumPy vs PyTorch

In [ ]:
def conv2d_numpy(X, W, b, stride=1, padding=0):
    N, C_in, H_in, W_in = X.shape
    C_out, _, Kh, Kw = W.shape
    H_out = int(np.floor((H_in - Kh + 2 * padding) / stride)) + 1
    W_out = int(np.floor((W_in - Kw + 2 * padding) / stride)) + 1
    X_pad = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')
    out = np.zeros((N, C_out, H_out, W_out), dtype=np.float32)
    for n in range(N):
        for c in range(C_out):
            for i in range(H_out):
                for j in range(W_out):
                    patch = X_pad[n, :, i*stride:i*stride+Kh, j*stride:j*stride+Kw]
                    out[n, c, i, j] = np.sum(patch * W[c]) + b[c]
    return out

# Buat tensor uji
X_np = np.random.randn(2, 3, 8, 8).astype(np.float32)
W_np = np.random.randn(4, 3, 3, 3).astype(np.float32)
b_np = np.random.randn(4).astype(np.float32)

# Hasil NumPy
out_np = conv2d_numpy(X_np, W_np, b_np, stride=1, padding=1)

# Hasil PyTorch
conv_torch = nn.Conv2d(3, 4, kernel_size=3, stride=1, padding=1)
conv_torch.weight.data = torch.tensor(W_np)
conv_torch.bias.data = torch.tensor(b_np)
out_pt = conv_torch(torch.tensor(X_np)).detach().numpy()

diff = np.max(np.abs(out_np - out_pt))
print(f"Selisih Absolut Maksimum NumPy vs PyTorch: {diff:.6e}")
assert diff < 1e-5, "Implementasi konvolusi tidak cocok!"
print("[VALIDASI SUKSES] Logika konvolusi 2D konsisten secara numerik!")

### Bagian 2: Sintesis Dataset Patologi Daun Kelapa Sawit
Membuat 300 sampel citra sintetis ($3 \times 64 \times 64$):
* Kelas 0: Daun Sehat (Hijau seragam dengan variasi tekstur serat alami).
* Kelas 1: Daun Terinfeksi Bercak Daun (*Curvularia*) (Bercak nekrotik cokelat gelap dengan halo kuning).

In [ ]:
def generate_palm_leaf_dataset(num_samples=300):
    X = np.zeros((num_samples, 3, 64, 64), dtype=np.float32)
    y = np.zeros(num_samples, dtype=np.int64)
    
    for i in range(num_samples):
        # Latar daun hijau tropis
        base_green = np.random.uniform(0.5, 0.8)
        X[i, 0, :, :] = np.random.uniform(0.1, 0.25) # Kanal R (rendah)
        X[i, 1, :, :] = base_green + np.random.normal(0, 0.05, (64, 64)) # Kanal G (tinggi)
        X[i, 2, :, :] = np.random.uniform(0.1, 0.3) # Kanal B (rendah)
        
        if i >= num_samples // 2:
            # Daun Sakit: Suntikkan bercak lesi nekrotik
            y[i] = 1
            num_spots = np.random.randint(3, 8)
            for _ in range(num_spots):
                cx, cy = np.random.randint(10, 54, size=2)
                radius = np.random.randint(3, 7)
                yy, xx = np.ogrid[:64, :64]
                dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
                mask_spot = dist <= radius
                # Lesi cokelat gelap / nekrotik
                X[i, 0, mask_spot] = 0.45
                X[i, 1, mask_spot] = 0.20
                X[i, 2, mask_spot] = 0.05
                
    # Klip ke rentang valid [0, 1]
    X = np.clip(X, 0.0, 1.0)
    return torch.tensor(X), torch.tensor(y)

X_all, y_all = generate_palm_leaf_dataset(400)
# Bagi Train (80%) dan Validation (20%)
train_ds = TensorDataset(X_all[:320], y_all[:320])
val_ds = TensorDataset(X_all[320:], y_all[320:])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"Dataset siap: Train = {len(train_ds)} sampel, Val = {len(val_ds)} sampel.")

### Bagian 3: Membangun dan Melatih Arsitektur CNN

In [ ]:
class AgroPalmCNN(nn.Module):
    def __init__(self):
        super(AgroPalmCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )
        
    def forward(self, x):
        return self.classifier(self.features(x))

model = AgroPalmCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Hitung parameter
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameter Model: {total_params:,}")

epochs = 10
train_losses, val_accs = [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    # Evaluasi
    model.eval()
    correct = 0
    with torch.no_grad():
        for val_x, val_y in val_loader:
            preds = model(val_x).argmax(dim=1)
            correct += (preds == val_y).sum().item()
    acc = correct / len(val_loader.dataset)
    val_accs.append(acc)
    
    print(f"Epoch [{epoch+1:02d}/{epochs}] - Loss: {epoch_loss:.4f} | Val Accuracy: {acc*100:.2f}%")

# Simpan plot metrik
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, 'b-o', label='Training Loss')
plt.title('Dinamika Konvergensi Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(val_accs, 'g-s', label='Validation Accuracy')
plt.title('Akurasi Validasi Model')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.grid(True)
plt.tight_layout()
plt.savefig('praktikum_cnn_convergence_12_1.png', dpi=200)
plt.close()
print("[SUKSES] Grafik konvergensi disimpan sebagai 'praktikum_cnn_convergence_12_1.png'")